In [ ]:
%pip install -q syllables

In [ ]:
!pip install syllables
import nltk
nltk.download('punkt_tab')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd
import syllables
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize

In [ ]:
cmufile = "YOUR_DATA_DIRECTORY"


with open(cmufile, encoding = 'latin-1') as cmu:
  cmuLines = cmu.readlines()

syllableDict = {}
for aline in cmuLines[56:]:
  syllableCount =0
  for char in aline:
    if char.isdigit():
      syllableCount +=1
  #print(aline.split()[0], syllableCount)
  syllableDict[aline.split()[0].lower()] = syllableCount

import syllables as syllable_estimator

def getSyllables(word):
    word = word.lower()
    if word in syllableDict:
        return syllableDict[word]
    return syllable_estimator.estimate(word)

def isComplexWord(word):
    return getSyllables(word) >= 3



def isFuture(sentence):
  futureWords = ['will','shall','may','might','should','going','gonna']
  sentWords = word_tokenize(sentence)
  print(sentWords)
  for word in futureWords:
    if word in sentWords:
      print(sentWords)
      return True
    else:
      return False



In [ ]:

text = "NLTK is gonna be a great NLP toolkit. It makes processing text easy!"
sentences = sent_tokenize(text)
#words = word_tokenize(sentences[0])
#print(words)
print(sentences[0])
print (isFuture(sentences[0]))


In [ ]:
# ---------- 변수 계산 ----------
def getWords(text):
  # 구두점, 숫자, "n't" 같은 조각은 제외하고 단어만
  return [w for w in word_tokenize(text) if w.replace('-', '').isalpha()]

def textFeatures(sentences):
  words = [w for s in sentences for w in getWords(s)]
  nSent = len(sentences)
  nWords = len(words)
  nComplex = sum(isComplexWord(w) for w in words)

  if nSent > 0 and nWords > 0:
    fog = 0.4 * ((nWords / nSent) + 100 * (nComplex / nWords))
  else:
    fog = float('nan')   # 미래 문장이 하나도 없는 문서

  feats = {
      'total_sentences': nSent,
      'total_words': nWords,
      'complex_words': nComplex,
      'fog': fog,
  }

  # 이전 주차 변수들도 여기 추가 (예시)
  # feats['positivity'] = getPositivity(' '.join(sentences))
  return feats

def analyzeDocument(text):
  sentences = sent_tokenize(text)
  futureSentences = [s for s in sentences if isFuture(s)]

  row = textFeatures(sentences)
  for key, value in textFeatures(futureSentences).items():
    row['future_' + key] = value     # future_total_sentences = 미래 문장 총 개수
  return row

In [ ]:
text = "NLTK is gonna be a great NLP toolkit. It makes processing text easy! We will analyze complicated documents tomorrow."
for s in sent_tokenize(text):
  print(isFuture(s), '|', s)

print(analyzeDocument(text))

In [ ]:
def analyzeDocument(text):
    if not isinstance(text, str):
        text = ""

    sentences = [
        sentence
        for sentence in sent_tokenize(text)
        if getWords(sentence)
    ]

    future_sentences = [
        sentence
        for sentence in sentences
        if isFuture(sentence)
    ]


    results = analyzeDocument(sentences)


    future_results = analyzeDocument(future_sentences)

    results.update({
        f"future_{name}": value
        for name, value in future_results.items()
    })


    results["total_future_sentences"] = len(future_sentences)

    return results


In [ ]:
import pandas as pd
df = pd.DataFrame({"text": [text]})